# 2PC Perf
调试2PC(CHEETAH)协议中的参数，对比性能

1. 128in, 8out
2. 256in, 256out
3. NLP precision

## (1) Load Model & Weights from HuggingFace

In [1]:
import jax

from flax_rnn import load_from_cache, generate, generate_topk, generate_greedy, generate_minp

# base_path = "/root/.cache/huggingface/hub/models--state-spaces--mamba-130m-hf/snapshots/1e76775f628fbf1350fbe4dbb3d971ba64af25a1"
base_path = "/root/shared-nvme/hf_cache/hub/models--state-spaces--mamba-130m-hf/snapshots/1e76775f628fbf1350fbe4dbb3d971ba64af25a1"
model, params, tokenizer = load_from_cache(base_path)

print("model loaded")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


model loaded


In [2]:
def generate_demo(prompt, gen_len=10, seed=42):
    input_ids = tokenizer.encode(prompt, return_tensors='jax')
    output_ids = generate_minp(model, params, input_ids, gen_len, seed=seed)
    print(prompt, tokenizer.decode(output_ids[0]), sep='')

In [3]:
generate_demo('Python is')

Python is in a bad state, and the fact that you


## (2) 定义运行函数

In [4]:
# 目的是规避model参数，给函数加上jit
# 重要：topk非常非常慢，而且运行时间和topk的值线性相关
@jax.jit
def gen_spu_greedy(params, input_ids):
    return generate_greedy(model, params, input_ids, n_tokens_to_gen=3) # 贪心采样，最快

@jax.jit
def gen_spu_topk(params, input_ids):
    return generate_topk(model, params, input_ids, n_tokens_to_gen=3, top_k=40) # topk采样，最慢

@jax.jit
def gen_spu_minp(params, input_ids):
    return generate_minp(model, params, input_ids, n_tokens_to_gen=3, min_p=0.1) # minp采样，比贪心慢一点点，但效果很好

# @jax.jit
# def gen_spu(params, input_ids):
#     # return generate(model, params, input_ids, n_tokens_to_gen=3) # 老代码
#     # return generate_greedy(model, params, input_ids, n_tokens_to_gen=3) # 贪心采样，最快
#     # return generate_topk(model, params, input_ids, n_tokens_to_gen=3, top_k=40) # topk采样，最慢
#     return generate_minp(model, params, input_ids, n_tokens_to_gen=3, min_p=0.1) # minp采样，比贪心慢一点点，但效果很好

## (3) SPU下验证性能
### 3.1 定义emulator

In [5]:
import sml.utils.emulation as emulation

mode = emulation.Mode.MULTIPROCESS

# note: in MULTIPROCESS mode, bandwidth and latency doesn't work
# emulation.CLUSTER_ABY3_3PC is a hard-coded string
# we copied it to current folder
emulator = emulation.Emulator(
    "2pc.json",
    mode,
    bandwidth=100,
    latency=10
)

emulator.up()

[2026-04-27 10:50:27,127]-[INFO]-[emulation.py:112]: Start multiprocess cluster...
[2026-04-27 10:50:27,721] [ForkServerProcess-3] Starting grpc server at 127.0.0.1:61922
[2026-04-27 10:50:27,721] [ForkServerProcess-1] Starting grpc server at 127.0.0.1:61920
[2026-04-27 10:50:27,721] [ForkServerProcess-5] Starting grpc server at 127.0.0.1:61924
[2026-04-27 10:50:27,721] [ForkServerProcess-2] Starting grpc server at 127.0.0.1:61921
[2026-04-27 10:50:27,721] [ForkServerProcess-4] Starting grpc server at 127.0.0.1:61923
[2026-04-27 10:50:29,209] [ForkServerProcess-2] Run : builtin_spu_init at node:1
[2026-04-27 10:50:29,209] [ForkServerProcess-3] Run : builtin_spu_init at node:2
[2026-04-27 10:50:29,209] [ForkServerProcess-1] Run : builtin_spu_init at node:0
I0427 10:50:29.228673 11327     0 external/brpc~/src/brpc/server.cpp:1195] Server[yacl::link::transport::internal::ReceiverServiceImpl] is serving on port=61932.
W0427 10:50:29.228689 11327     0 external/brpc~/src/brpc/server.cpp:120

### 3.2 greedy

In [ ]:
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_spu_greedy)(s_params, s_input_ids)
# don't print output here, profiler output may mess up

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
[2026-04-27 10:50:31,560] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-27 10:50:31,611] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-27 10:50:31,617] [ForkServerProcess-4] Run : make_shares at node:3


[2026-04-27 10:50:31.618] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95


[2026-04-27 10:50:37,769] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 10:50:37,773] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 10:50:37,778] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 10:50:37,781] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 10:50:37,782] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 10:50:37,785] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 10:50:37,786] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 10:50:37,788] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 10:50:37,790] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 10:50:37,793] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 10:50:37,794] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 10:50:37,796] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 10:50:37,804] [ForkServerProcess-4

[2026-04-27 10:51:10.774] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-27 10:51:10.775] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-27 10:51:10.776] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95


In [ ]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

### 3.3 minp

In [ ]:
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_spu_minp)(s_params, s_input_ids)
# don't print output here, profiler output may mess up

In [ ]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

### 3.4 topk

In [ ]:
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_spu_topk)(s_params, s_input_ids)
# don't print output here, profiler output may mess up

In [ ]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

## (4) cleanup

In [ ]:
emulator.down()